In [1]:
import os
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import webdataset as wds
import json
import time
import csv
from pathlib import Path
from tqdm.notebook import tqdm

# ==========================================
# 1. HARDWARE & PATH INITIALIZATION
# ==========================================
def select_gpu():
    """Prints live GPU stats and requires manual user selection."""
    print("\n[SYSTEM] Live GPU Status:")
    # Run nvidia-smi directly in the console so the user can read it
    subprocess.run(["nvidia-smi"])
    
    while True:
        gpu_id = input("\n[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']: ").strip()
        
        if not gpu_id:
            print("[SYSTEM] Defaulting to GPU 0.")
            return "0"
            
        if gpu_id.isdigit():
            print(f"[SYSTEM] Manually locking script to GPU {gpu_id}.")
            return gpu_id
            
        print("[!] Invalid input. Please enter a valid integer ID.")

# 1. LOCK THE GPU BEFORE INITIALIZING OTHER LOGIC
os.environ["CUDA_VISIBLE_DEVICES"] = select_gpu()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f"[SYSTEM] PyTorch initialized. Active Device: {torch.cuda.get_device_name(0)}")
else:
    print(f"[WARNING] PyTorch could not find a CUDA device. Falling back to CPU.")

user_prefix = input("\nEnter trial prefix (e.g., tcn_v1) [Press Enter for default 'unknown_trial']: ").strip()
TRIAL_PREFIX = user_prefix if user_prefix else "unknown_trial"

PROJECT_DIR = Path(os.path.expanduser("~/Capstone/Implementation"))
LOCAL_SHARDS_DIR = PROJECT_DIR / "data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards"
SAVE_DIR = PROJECT_DIR / f"models/{TRIAL_PREFIX}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SCALARS_PATH = LOCAL_SHARDS_DIR / "normalization_scalars.json"

with open(SCALARS_PATH, 'r') as f:
    scalars = json.load(f)

mins = torch.tensor([scalars['global_minimums'][c] for c in scalars['feature_order']], device=device)
maxs = torch.tensor([scalars['global_maximums'][c] for c in scalars['feature_order']], device=device)
ranges = maxs - mins
ranges[ranges == 0] = 1.0

# ==========================================
# 2. ARCHITECTURE & DATALOADERS
# ==========================================
BATCH_SIZE = 16384
TOTAL_TRAIN_BATCHES = 10407 
TOTAL_VAL_BATCHES = 1198
NUM_WORKERS = 8

IDX_P0, IDX_P1 = 0, 1
IDX_T0, IDX_T1 = 4, 5

EMPIRICAL = {
    "C0": 143.22, "C1": 140.25, "h0": 4.8713, "h1": 5.3871,
    "k01": 0.078038, "k10": 0.028120, "q0": 15.0, "q1": 15.0
}

class DilatedTCN(nn.Module):
    def __init__(self, num_inputs=6, num_channels=[32, 64, 128], kernel_size=3):
        super(DilatedTCN, self).__init__()
        layers = []
        in_channels = num_inputs
        for i, out_channels in enumerate(num_channels):
            dilation_size = 2 ** i
            padding = (kernel_size - 1) * dilation_size // 2
            layers += [
                nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding, dilation=dilation_size),
                nn.ReLU(),
                nn.BatchNorm1d(out_channels)
            ]
            in_channels = out_channels
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], 2)

    def forward(self, x):
        x = x.transpose(1, 2)
        out = self.tcn(x)
        return self.fc(out[:, :, -1])

class PINNEngine(nn.Module):
    def __init__(self):
        super(PINNEngine, self).__init__()
        self.tcn = DilatedTCN()
        self.raw_params = nn.ParameterDict({k: nn.Parameter(torch.tensor(0.0)) for k in EMPIRICAL.keys()})
        self.T_amb = 30.5

    def get_bounded_physics(self):
        phys = {}
        for k, exact_val in EMPIRICAL.items():
            min_bound = exact_val * 0.90
            max_bound = exact_val * 1.10
            phys[k] = min_bound + (max_bound - min_bound) * torch.sigmoid(self.raw_params[k])
        return phys

    def forward(self, x):
        return self.tcn(x)

def make_windows_vectorized(src, window_size=50, stride=10):
    for key, npy_array in src:
        n_rows = npy_array.shape[0]
        if n_rows < window_size: continue
        windows = np.lib.stride_tricks.sliding_window_view(npy_array, window_shape=window_size, axis=0)
        windows = np.swapaxes(windows, 1, 2)
        windows = windows[::stride]
        for w in windows: yield key, w

def create_dataloader(shard_list, is_train=True):
    shard_shuffle_val = 100 if is_train else 0
    dataset = wds.WebDataset(shard_list, shardshuffle=shard_shuffle_val).decode().to_tuple("__key__", "data.npy")
    dataset = dataset.compose(lambda src: make_windows_vectorized(src, window_size=50, stride=10))
    if is_train: dataset = dataset.shuffle(5000)
    dataset = dataset.batched(BATCH_SIZE)
    return torch.utils.data.DataLoader(dataset, batch_size=None, num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=2)

train_files = [str(p) for p in sorted((LOCAL_SHARDS_DIR / "train").glob("*.tar"))]
val_files = [str(p) for p in sorted((LOCAL_SHARDS_DIR / "val").glob("*.tar"))]

train_loader = create_dataloader(train_files, is_train=True)
val_loader = create_dataloader(val_files, is_train=False)

# ==========================================
# 3. CALIBRATION INFRASTRUCTURE (5 RUNS)
# ==========================================
EPOCHS = 1             # FORCED to 1 for calibration
LEARNING_RATE = 1e-3
LAMBDA_DATA = 1.0      
LAMBDA_PHYS = 1.0      # Set to 1.0 during calibration so it doesn't skew raw averages
DT = 0.1

print(f"\n[TRAINING] INITIATING 5-RUN CALIBRATION FOR LAMBDA_PHYS [{TRIAL_PREFIX}]")

# Data structures to hold the results of our 5 runs
calibration_data_losses = []
calibration_phys_losses = []

try:
    for run in range(1, 6):
        print(f"\n" + "="*80)
        print(f"[CALIBRATION] STARTING RUN {run}/5")
        print("="*80)

        # RE-INITIALIZE EVERYTHING INSIDE THE LOOP
        # This guarantees fresh random weights for every single run.
        model = PINNEngine().to(device)
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-6)
        mse_loss = nn.MSELoss()
        scaler = torch.amp.GradScaler('cuda')
        
        start_epoch = 1
        best_val_loss = float('inf')
        
        # Append run number to paths so they don't overwrite
        current_prefix = f"{TRIAL_PREFIX}_run{run}"
        RESUME_PATH = SAVE_DIR / f"{current_prefix}_resume_checkpoint.pt"
        BEST_PATH = SAVE_DIR / f"{current_prefix}_best_model.pt"
        gpu_log_path = SAVE_DIR / f"{current_prefix}_gpu_stats.csv"
        epoch_log_path = SAVE_DIR / f"{current_prefix}_epoch_logs.csv"

        # Logging initialization for this specific run
        gpu_log_file = open(gpu_log_path, "w")
        gpu_logger = subprocess.Popen(
            ["nvidia-smi", "--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,temperature.gpu", "--format=csv", "-l", "5"],
            stdout=gpu_log_file, stderr=subprocess.STDOUT
        )

        csv_mode = 'a' if RESUME_PATH.exists() and epoch_log_path.exists() else 'w'
        epoch_log_file = open(epoch_log_path, mode=csv_mode, newline='')
        epoch_writer = csv.writer(epoch_log_file)
        if csv_mode == 'w':
            epoch_writer.writerow([
                'Epoch', 'Train_Loss_Total', 'Train_Loss_Data', 'Train_Loss_Phys',
                'Val_Loss_Total', 'Val_Loss_Data', 'Val_Loss_Phys',
                'C0', 'C1', 'h0', 'h1', 'k01', 'k10', 'q0', 'q1'
            ])

        # ==========================================
        # 4. MAIN EPOCH LOOP (LOCKED TO 1 EPOCH)
        # ==========================================
        for epoch in range(start_epoch, EPOCHS + 1):
            epoch_start_time = time.time()
            model.train()
            
            total_train_loss, total_train_loss_data, total_train_loss_phys = 0.0, 0.0, 0.0
            train_pbar = tqdm(train_loader, total=TOTAL_TRAIN_BATCHES, desc=f"Run {run} - Epoch {epoch:02d}/{EPOCHS} [Train]", leave=False)

            for batch_idx, (keys, data_block) in enumerate(train_pbar):
                X_seq = data_block[:, :-1, :].to(device)
                Y_true_abs = data_block[:, -1, 4:6].to(device)
                Y_true_norm = (Y_true_abs - mins[4:6]) / ranges[4:6]
                X_norm = (X_seq - mins) / ranges

                optimizer.zero_grad()

                with torch.amp.autocast('cuda'):
                    Y_pred_norm = model(X_norm)
                    
                    # DATA LOSS (NORMALIZED)
                    loss_data = mse_loss(Y_pred_norm, Y_true_norm)
                    
                    # PHYSICS EXTRACTION (ABSOLUTE)
                    Y_pred_abs = Y_pred_norm * ranges[4:6] + mins[4:6]
                    X_last_step = X_seq[:, -1, :]
                    P0_abs, P1_abs = X_last_step[:, IDX_P0], X_last_step[:, IDX_P1]
                    T0_t_abs, T1_t_abs = X_last_step[:, IDX_T0], X_last_step[:, IDX_T1]
                    T0_pred_abs, T1_pred_abs = Y_pred_abs[:, 0], Y_pred_abs[:, 1]
                
                    phys = model.get_bounded_physics()
                    res_0 = ((T0_pred_abs - T0_t_abs) / DT) - (1/phys["C0"]) * (P0_abs + phys["k01"]*P1_abs - phys["h0"]*(T0_t_abs - model.T_amb) + phys["q0"])
                    res_1 = ((T1_pred_abs - T1_t_abs) / DT) - (1/phys["C1"]) * (P1_abs + phys["k10"]*P0_abs - phys["h1"]*(T1_t_abs - model.T_amb) + phys["q1"])
                
                    # PHYSICS LOSS (ABSOLUTE)
                    zeros = torch.zeros_like(res_0)
                    loss_phys = mse_loss(res_0, zeros) + mse_loss(res_1, zeros)
                    
                    # COMPOSITE LOSS
                    loss_total = (LAMBDA_DATA * loss_data) + (LAMBDA_PHYS * loss_phys)

                scaler.scale(loss_total).backward()
                scaler.step(optimizer)
                scaler.update()

                total_train_loss += loss_total.item()
                total_train_loss_data += loss_data.item()
                total_train_loss_phys += loss_phys.item()
                train_pbar.set_postfix({"Loss": f"{loss_total.item():.4f}"})

            avg_train_loss = total_train_loss / TOTAL_TRAIN_BATCHES
            avg_train_data = total_train_loss_data / TOTAL_TRAIN_BATCHES
            avg_train_phys = total_train_loss_phys / TOTAL_TRAIN_BATCHES
            
            # STORE AVERAGES FOR LAMBDA CALCULATION
            calibration_data_losses.append(avg_train_data)
            calibration_phys_losses.append(avg_train_phys)

            # ==========================================
            # VALIDATION LOOP
            # ==========================================
            model.eval()
            total_val_loss, total_val_loss_data, total_val_loss_phys = 0.0, 0.0, 0.0
            val_pbar = tqdm(val_loader, total=TOTAL_VAL_BATCHES, desc=f"Run {run} - Validation", leave=False)

            with torch.no_grad():
                for batch_idx, (keys, val_block) in enumerate(val_pbar):
                    X_val = val_block[:, :-1, :].to(device)
                    Y_val_true_abs = val_block[:, -1, 4:6].to(device)
                    Y_val_true_norm = (Y_val_true_abs - mins[4:6]) / ranges[4:6]
                    X_val_norm = (X_val - mins) / ranges
                    
                    with torch.amp.autocast('cuda'):
                        Y_val_pred_norm = model(X_val_norm)
                        loss_data_v = mse_loss(Y_val_pred_norm, Y_val_true_norm)

                        Y_val_pred_abs = Y_val_pred_norm * ranges[4:6] + mins[4:6]
                        X_val_last_step = X_val[:, -1, :]

                        P0_abs_v, P1_abs_v = X_val_last_step[:, IDX_P0], X_val_last_step[:, IDX_P1]
                        T0_t_abs_v, T1_t_abs_v = X_val_last_step[:, IDX_T0], X_val_last_step[:, IDX_T1]
                        T0_pred_abs_v, T1_pred_abs_v = Y_val_pred_abs[:, 0], Y_val_pred_abs[:, 1]

                        phys_v = model.get_bounded_physics()
                        res_0_v = ((T0_pred_abs_v - T0_t_abs_v) / DT) - (1/phys_v["C0"]) * (P0_abs_v + phys_v["k01"]*P1_abs_v - phys_v["h0"]*(T0_t_abs_v - model.T_amb) + phys_v["q0"])
                        res_1_v = ((T1_pred_abs_v - T1_t_abs_v) / DT) - (1/phys_v["C1"]) * (P1_abs_v + phys_v["k10"]*P0_abs_v - phys_v["h1"]*(T1_t_abs_v - model.T_amb) + phys_v["q1"])

                        zeros_v = torch.zeros_like(res_0_v)
                        loss_phys_v = mse_loss(res_0_v, zeros_v) + mse_loss(res_1_v, zeros_v)
                        loss_total_v = (LAMBDA_DATA * loss_data_v) + (LAMBDA_PHYS * loss_phys_v)
                        
                        total_val_loss += loss_total_v.item()
                        total_val_loss_data += loss_data_v.item()
                        total_val_loss_phys += loss_phys_v.item()

            avg_val_loss = total_val_loss / TOTAL_VAL_BATCHES
            avg_val_data = total_val_loss_data / TOTAL_VAL_BATCHES
            avg_val_phys = total_val_loss_phys / TOTAL_VAL_BATCHES
            epoch_duration = time.time() - epoch_start_time

            # CSV Logging
            current_phys = model.get_bounded_physics()
            epoch_writer.writerow([
                epoch, avg_train_loss, avg_train_data, avg_train_phys,
                avg_val_loss, avg_val_data, avg_val_phys,
                current_phys['C0'].item(), current_phys['C1'].item(),
                current_phys['h0'].item(), current_phys['h1'].item(),
                current_phys['k01'].item(), current_phys['k10'].item(),
                current_phys['q0'].item(), current_phys['q1'].item()
            ])
            epoch_log_file.flush()

            # MATHEMATICAL WALKTHROUGH (Printed for EVERY Run)
            print("\n[MATH] " + "="*80)
            print(f"[MATH] >>> RUN {run}: MATHEMATICAL WALKTHROUGH & PROOF <<<")
            print("[MATH] " + "="*80)
            print("[MATH] --- PART 1: AGGREGATE TRAINING LOSS PROOF ---")
            print(f"[MATH] LAMBDA_DATA: {LAMBDA_DATA} | LAMBDA_PHYS: {LAMBDA_PHYS}")
            print(f"[MATH] Avg Train Data Loss : {avg_train_data:.8f}")
            print(f"[MATH] Avg Train Phys Loss : {avg_train_phys:.8f}")
            calc_train = (LAMBDA_DATA * avg_train_data) + (LAMBDA_PHYS * avg_train_phys)
            print(f"[MATH] Math: ({LAMBDA_DATA} * {avg_train_data:.8f}) + ({LAMBDA_PHYS} * {avg_train_phys:.8f}) = {calc_train:.8f}")
            print(f"[MATH] Printed Train Loss  : {avg_train_loss:.8f}")
            
            print("\n[MATH] --- PART 2: AGGREGATE VALIDATION LOSS PROOF ---")
            print(f"[MATH] Avg Val Data Loss   : {avg_val_data:.8f}")
            print(f"[MATH] Avg Val Phys Loss   : {avg_val_phys:.8f}")
            calc_val = (LAMBDA_DATA * avg_val_data) + (LAMBDA_PHYS * avg_val_phys)
            print(f"[MATH] Math: ({LAMBDA_DATA} * {avg_val_data:.8f}) + ({LAMBDA_PHYS} * {avg_val_phys:.8f}) = {calc_val:.8f}")
            print(f"[MATH] Printed Val Loss    : {avg_val_loss:.8f}")
            
            print("\n[MATH] --- PART 3: ROW-LEVEL PHYSICS RESIDUAL DISSECTION ---")
            print("[MATH] Extracting Row 0 from the final Validation batch:")
            
            p0, p1 = P0_abs_v[0].item(), P1_abs_v[0].item()
            t0_t, t1_t = T0_t_abs_v[0].item(), T1_t_abs_v[0].item()
            t0_p, t1_p = T0_pred_abs_v[0].item(), T1_pred_abs_v[0].item()
            
            print(f"[MATH] Inputs   -> P0: {p0:.4f}, P1: {p1:.4f}")
            print(f"[MATH] Targets  -> T0_t: {t0_t:.4f}, T1_t: {t1_t:.4f}")
            print(f"[MATH] Predicts -> T0_pred: {t0_p:.4f}, T1_pred: {t1_p:.4f}")
            
            print("\n[MATH] Equation res_0:")
            print(f"[MATH] res_0 = (({t0_p:.4f} - {t0_t:.4f}) / {DT}) - (1/{current_phys['C0'].item():.4f}) * ({p0:.4f} + {current_phys['k01'].item():.4f}*{p1:.4f} - {current_phys['h0'].item():.4f}*({t0_t:.4f} - {model.T_amb}) + {current_phys['q0'].item():.4f})")
            r0_val = res_0_v[0].item()
            print(f"[MATH] Actual Tensor Output (res_0) = {r0_val:.6f}")
            
            print("\n[MATH] Equation res_1:")
            print(f"[MATH] res_1 = (({t1_p:.4f} - {t1_t:.4f}) / {DT}) - (1/{current_phys['C1'].item():.4f}) * ({p1:.4f} + {current_phys['k10'].item():.4f}*{p0:.4f} - {current_phys['h1'].item():.4f}*({t1_t:.4f} - {model.T_amb}) + {current_phys['q1'].item():.4f})")
            r1_val = res_1_v[0].item()
            print(f"[MATH] Actual Tensor Output (res_1) = {r1_val:.6f}")
            
            print("\n[MATH] --- PART 4: THE SQUARED PENALTY (MSE) ---")
            print(f"[MATH] res_0 squared : ({r0_val:.6f})^2 = {r0_val**2:.6f}")
            print(f"[MATH] res_1 squared : ({r1_val:.6f})^2 = {r1_val**2:.6f}")
            print(f"[MATH] Total Row Physics Penalty : {(r0_val**2) + (r1_val**2):.6f}")
            print("[MATH] " + "="*80 + "\n")

            print(f"[CALIBRATION] Run {run}/5 | Time: {epoch_duration:.1f}s | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

        # Clean up loggers for this run before starting the next
        gpu_logger.terminate()
        gpu_log_file.close()
        epoch_log_file.close()

    # ==========================================
    # FINAL CALCULATION: THE MAGIC LAMBDA
    # ==========================================
    print("\n" + "#"*80)
    print("### CALIBRATION COMPLETE: CALCULATING OPTIMAL LAMBDA_PHYS ###")
    print("#"*80)
    
    for i in range(5):
        print(f"Run {i+1} -> Data Loss: {calibration_data_losses[i]:.8f} | Phys Loss: {calibration_phys_losses[i]:.4f}")
        
    final_avg_data = sum(calibration_data_losses) / 5.0
    final_avg_phys = sum(calibration_phys_losses) / 5.0
    
    optimal_lambda = final_avg_data / final_avg_phys
    
    print("\n--- FINAL AVERAGES ---")
    print(f"Average Data Loss: {final_avg_data:.8f}")
    print(f"Average Phys Loss: {final_avg_phys:.4f}")
    
    print("\n--- THE RESULT ---")
    print(f"Optimal LAMBDA_PHYS = ({final_avg_data:.8f} / {final_avg_phys:.4f})")
    print(f"Optimal LAMBDA_PHYS = {optimal_lambda:.8e}")
    print("\n[SYSTEM] Update your original script with this LAMBDA_PHYS value for full training!")

except KeyboardInterrupt:
    print("\n[!] Calibration manually interrupted.")
    gpu_logger.terminate()
    gpu_log_file.close()
    epoch_log_file.close()


[SYSTEM] Live GPU Status:
Tue Mar 24 09:47:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500               Off |   00000000:3B:00.0 Off |                  Off |
| 30%   36C    P8              8W /  200W |   11395MiB /  20470MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------


[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']:  1


[SYSTEM] Manually locking script to GPU 1.
[SYSTEM] PyTorch initialized. Active Device: NVIDIA RTX A4500



Enter trial prefix (e.g., tcn_v1) [Press Enter for default 'unknown_trial']:  lambda_physics_calculation



[TRAINING] INITIATING 5-RUN CALIBRATION FOR LAMBDA_PHYS [lambda_physics_calculation]

[CALIBRATION] STARTING RUN 1/5


...

Run 4 - Validation:   0%|          | 0/1198 [00:00<?, ?it/s]


[MATH] ================================================================================
[MATH] >>> RUN 4: MATHEMATICAL WALKTHROUGH & PROOF <<<
[MATH] ================================================================================
[MATH] --- PART 1: AGGREGATE TRAINING LOSS PROOF ---
[MATH] LAMBDA_DATA: 1.0 | LAMBDA_PHYS: 1.0
[MATH] Avg Train Data Loss : 0.00496130
[MATH] Avg Train Phys Loss : 3663.37727163
[MATH] Math: (1.0 * 0.00496130) + (1.0 * 3663.37727163) = 3663.38223293
[MATH] Printed Train Loss  : 3663.38222588

[MATH] --- PART 2: AGGREGATE VALIDATION LOSS PROOF ---
[MATH] Avg Val Data Loss   : 0.07878961
[MATH] Avg Val Phys Loss   : 58522.34187803
[MATH] Math: (1.0 * 0.07878961) + (1.0 * 58522.34187803) = 58522.42066764
[MATH] Printed Val Loss    : 58522.42081210

[MATH] --- PART 3: ROW-LEVEL PHYSICS RESIDUAL DISSECTION ---
[MATH] Extracting Row 0 from the final Validation batch:
[MATH] Inputs   -> P0: 42.1200, P1: 43.4900
[MATH] Targets  -> T0_t: 37.0000, T1_t: 35.0000
[MATH

Run 5 - Epoch 01/1 [Train]:   0%|          | 0/10407 [00:00<?, ?it/s]

Run 5 - Validation:   0%|          | 0/1198 [00:00<?, ?it/s]


[MATH] ================================================================================
[MATH] >>> RUN 5: MATHEMATICAL WALKTHROUGH & PROOF <<<
[MATH] ================================================================================
[MATH] --- PART 1: AGGREGATE TRAINING LOSS PROOF ---
[MATH] LAMBDA_DATA: 1.0 | LAMBDA_PHYS: 1.0
[MATH] Avg Train Data Loss : 0.00550032
[MATH] Avg Train Phys Loss : 4059.13432365
[MATH] Math: (1.0 * 0.00550032) + (1.0 * 4059.13432365) = 4059.13982398
[MATH] Printed Train Loss  : 4059.13982389

[MATH] --- PART 2: AGGREGATE VALIDATION LOSS PROOF ---
[MATH] Avg Val Data Loss   : 0.01275947
[MATH] Avg Val Phys Loss   : 9399.23426677
[MATH] Math: (1.0 * 0.01275947) + (1.0 * 9399.23426677) = 9399.24702624
[MATH] Printed Val Loss    : 9399.24703513

[MATH] --- PART 3: ROW-LEVEL PHYSICS RESIDUAL DISSECTION ---
[MATH] Extracting Row 0 from the final Validation batch:
[MATH] Inputs   -> P0: 42.1200, P1: 43.4900
[MATH] Targets  -> T0_t: 37.0000, T1_t: 35.0000
[MATH] Pr